In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Importing dataset
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [ ]:
# Handling categorical data
dataset = pd.get_dummies(dataset,drop_first=True,dtype=int)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [ ]:
# Dropping the 'USER ID' Columns as it is unique identifier
dataset = dataset.drop("User ID",axis=1)
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [5]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [6]:
dataset['Purchased'].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [7]:
# Feature and target selection
independent = dataset[['Age', 'EstimatedSalary', 'Gender_Male']]
dependent = dataset['Purchased']

In [8]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [9]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [10]:
# Splitting data into training and test sets
x_train,x_test,y_train,y_test = train_test_split(independent,dependent,test_size=0.3,random_state=0)

In [11]:
x_train.shape

(280, 3)

In [12]:
x_test.shape

(120, 3)

In [13]:
y_train.shape

(280,)

In [14]:
y_test.shape

(120,)

In [ ]:
# Data Standardization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [ ]:
# GridSearch CV
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Defining the different parameters for gridsearch optimization
param_grid = {'solver':[ 'lbfgs','newton-cg', 'liblinear', 'saga'],'penalty':['l2']}
grid = GridSearchCV(LogisticRegression(),param_grid,refit=True,verbose=3,scoring='f1_weighted',n_jobs=-1)
grid.fit(x_train,y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


GridSearchCV(estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'penalty': ['l2'],
                         'solver': ['lbfgs', 'newton-cg', 'liblinear', 'saga']},
             scoring='f1_weighted', verbose=3)

In [ ]:
# Best gridsearch parameter
print(grid.best_params_)

{'penalty': 'l2', 'solver': 'lbfgs'}


In [ ]:
# Grid prediction
y_pred = grid.predict(x_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 0, 1, 1])

In [30]:
# evaluation metrics
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[74  5]
 [ 8 33]]


In [31]:
from sklearn.metrics import classification_report
report = classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.90      0.94      0.92        79
           1       0.87      0.80      0.84        41

    accuracy                           0.89       120
   macro avg       0.89      0.87      0.88       120
weighted avg       0.89      0.89      0.89       120



In [33]:
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test,y_pred,average='weighted')
print(f1_macro)

0.8906190214115365


In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

np.float64(0.9481321395492436)

In [36]:
age = int(input("Enter your age:"))
salary = float(input("Enter your salary"))
gender = int(input("male=1,female=0"))

In [37]:
result = grid.predict([[age,salary,gender]])
result

array([1])